# Step 4: Label Calibration, Spatially Explicit Features and Robustness Checks

> **Parsing note**: `pandas.read_html` merges the line breaks between list items into spaces, so this notebook turns list items into semicolons before reading the Wikipedia table; otherwise a cooperative's municipalities collapse into one unmatched string. Spatial features are split into H+Sgeo (neighbouring wind, island size) and H+S (adds the cooperative features), and η² counts only groups with at least 2 municipalities.

Step 3 left four open problems. This step answers each one:

| Part | Problem | Approach |
| --- | --- | --- |
| A | Many far-field "outages" are noise | **Placebo calibration**: for each typhoon, lower the outage threshold step by step (80% → 40%) and choose the most lenient threshold whose placebo false-positive rate is ≤ 5%, then regenerate the labels |
| B | Residuals are strongly spatially clustered | Add a set of **spatially explicit features**: wind in neighbouring municipalities (spatial lag), the area of the island the municipality is on, and the overall storm burden and size of its electric cooperative's service territory |
| C | Uncertainty unknown; do the embeddings help at all? | **Bootstrap 95% confidence intervals** on every comparison; embeddings tested with 3 and with 10 principal components |
| D | The equity metric was not decision-relevant | Use a decision metric instead: **if crews prioritised the 30% of municipalities with the highest predicted risk, what share of truly long outages would they cover**; and compare whether different wealth groups can be measured by satellite at all |

**Source for cooperative service areas**: there is no official GIS file. This notebook reads Wikipedia's [List of electric distribution utilities in the Philippines](https://en.wikipedia.org/wiki/List_of_electric_distribution_utilities_in_the_Philippines) (each distribution utility's type, customer count and the municipalities it serves) and matches it to municipal boundaries by name and province. It is an unofficial compilation, and some municipalities are split between two utilities (marked "some areas only"); say so in the write-up. If automatic parsing fails, you can prepare a `utility_areas.csv` by hand and put it in Drive (format described in cell 7).

**No Earth Engine is needed in this step.** Everything comes from files already in Drive: `step2b_labels.csv`, `step3_features.csv`, `ntl_*.csv`, `meta_*.csv`, and the optional RWI file. The whole notebook takes about 10–15 minutes, mostly for the recomputation in Part A and the bootstrap in Part C.

In [ ]:
# 1. Install dependencies
!pip install -q geopandas scikit-learn lxml

In [ ]:
# 2. Mount Drive
import os, glob, io, re, unicodedata, warnings
from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER = "ph_typhoon_ntl"
DRIVE_DIR = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
os.makedirs("figures", exist_ok=True)
warnings.filterwarnings("ignore")
print("OK")

In [ ]:
# 3. Events, parameters and label functions (outage threshold is now an argument)
import numpy as np
import pandas as pd

EVENTS = pd.DataFrame([
    ("mangkhut_2018", "MANGKHUT", 2018, "Ompong",  "2018-09-15 01:40", None),
    ("kammuri_2019",  "KAMMURI",  2019, "Tisoy",   "2019-12-02 23:00", None),
    ("phanfone_2019", "PHANFONE", 2019, "Ursula",  "2019-12-24 16:45", "2019-11-28"),
    ("rai_2021",      "RAI",      2021, "Odette",  "2021-12-16 13:30", None),
], columns=["key", "ibtracs_name", "season", "local_name", "landfall", "baseline_end"])
EVENTS["landfall"] = pd.to_datetime(EVENTS["landfall"])
EV = EVENTS.set_index("key")
MODEL_KEYS = EVENTS.key.tolist()

BASE_DAYS, BASE_GAP_DAYS, POST_WEEKS = 120, 3, 26
MIN_VALID_FRAC, MOON_MAX_ILLUM = 0.5, 0.8
MIN_DAYS_PER_WEEK, MIN_BASE_NIGHTS = 2, 8
RECOVERY_LEVEL = 80
DAILY_DAYS, ROLL_N, MAX_SPAN_DAYS, FIRST_LOOK_DAYS = 28, 3, 6, 5
OVERPASS_LAG = pd.Timedelta(hours=25, minutes=30)
REF_NEW_MOON, SYNODIC = pd.Timestamp("2000-01-06 18:14"), 29.530588853
LONG_OUTAGE_DAYS = 7

# Part A: candidate outage thresholds and target false-positive rate
LEVELS = [80, 70, 60, 50, 40]
TARGET_FP = 0.05
# Part D: share of municipalities prioritised
PRIORITY_SHARE = 0.30
N_BOOT = 1000

def windows(ev):
    base_end = pd.Timestamp(ev.baseline_end) if isinstance(ev.baseline_end, str) else ev.landfall - pd.Timedelta(days=BASE_GAP_DAYS)
    return base_end - pd.Timedelta(days=BASE_DAYS), base_end

def moon_illum(obs_utc):
    age = ((obs_utc - REF_NEW_MOON) / pd.Timedelta(days=1)) % SYNODIC
    return (1 - np.cos(2 * np.pi * age / SYNODIC)) / 2

def load_event(key):
    files = [f for f in sorted(glob.glob(f"{DRIVE_DIR}/ntl_{key}*.csv")) if "_px4" not in os.path.basename(f)]
    if not files:
        return None
    d = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
    d["date"] = pd.to_datetime(d["date"])
    d["obs_pht"] = d["date"] + OVERPASS_LAG
    d["illum"] = moon_illum(d["obs_pht"] - pd.Timedelta(hours=8))
    return d

def week_of(obs, t0):
    return (np.ceil((obs - t0) / pd.Timedelta(days=1) / 7) - 1).astype(int)

def label_all(g, t0, base_start, base_end, censor_at, level=80):
    ok = g["mean"].notna() & (g["count"] >= MIN_VALID_FRAC * g["eligible_px"]) & (g["illum"] <= MOON_MAX_ILLUM)
    g = g[ok].sort_values("obs_pht")
    base = g[(g.obs_pht >= base_start) & (g.obs_pht <= base_end)]
    out = {"base_nights": len(base)}
    if len(base) < 3:
        return {**out, "status_w": "no_baseline", "status_d": "no_baseline"}
    b = base["mean"].median()
    g = g.assign(pct=g["mean"] / b * 100, wk=week_of(g.obs_pht, t0))
    end = t0 + pd.Timedelta(weeks=POST_WEEKS)
    if pd.notna(censor_at):
        end = min(end, censor_at)
    pre = g.loc[base.index].groupby("wk").pct.agg(["median", "size"])
    noise = pre.loc[pre["size"] >= MIN_DAYS_PER_WEEK, "median"].std()
    w = g[(g.obs_pht > t0) & (g.obs_pht < end)].groupby("wk").pct.agg(["median", "size"])
    w = w.loc[w["size"] >= MIN_DAYS_PER_WEEK, "median"]
    early = w[w.index <= 3]
    if early.empty:
        out["status_w"] = "no_early_data"
    else:
        low, low_wk = early.min(), early.idxmin()
        thr = level if pd.isna(noise) else min(level, 100 - 2 * noise)
        out.update(outage_w=bool(low < thr), status_w="ok" if len(base) >= MIN_BASE_NIGHTS else "low_baseline")
        if not out["outage_w"]:
            out.update(recovery_weeks=0, censored_w=False)
        else:
            after = w[w.index >= low_wk]
            hit = [k for k, a, c in zip(after.index[:-1], after.values[:-1], after.values[1:])
                   if a >= RECOVERY_LEVEL and c >= RECOVERY_LEVEL]
            out.update(recovery_weeks=int(hit[0]) + 1 if hit else int(w.index.max()) + 1, censored_w=not hit)
    bp = g.loc[base.index, "pct"].values
    rolls_b = [np.median(bp[i:i + ROLL_N]) for i in range(len(bp) - ROLL_N + 1)]
    noise3 = np.std(rolls_b, ddof=1) if len(rolls_b) > 2 else np.nan
    thr_d = level if pd.isna(noise3) else min(level, 100 - 2 * noise3)
    post = g[(g.obs_pht > t0) & (g.obs_pht < min(end, t0 + pd.Timedelta(days=DAILY_DAYS)))]
    days = ((post.obs_pht - t0) / pd.Timedelta(days=1)).values
    vals = post.pct.values
    rolls = [(days[i], np.median(vals[i:i + ROLL_N])) for i in range(len(vals) - ROLL_N + 1)
             if days[i + ROLL_N - 1] - days[i] <= MAX_SPAN_DAYS]
    if not rolls:
        out["status_d"] = "too_few_nights"
    elif rolls[0][0] > FIRST_LOOK_DAYS:
        out["status_d"] = "late_first_look"
    else:
        rd, rv = np.array([r[0] for r in rolls]), np.array([r[1] for r in rolls])
        ei = np.where(rd <= FIRST_LOOK_DAYS)[0]
        i_min = ei[np.argmin(rv[ei])]
        out.update(outage_d=bool(rv[i_min] < thr_d), status_d="ok")
        if not out["outage_d"]:
            out.update(outage_days=0.0, censored_d=False)
        else:
            rec = np.where((rv >= RECOVERY_LEVEL) & (np.arange(len(rv)) > i_min))[0]
            out.update(outage_days=float(rd[rec[0]]) if len(rec) else float(days[-1]), censored_d=not len(rec))
    return out

def combine(r):
    wk_ok = r.get("status_w") in ("ok", "low_baseline")
    d_ok = r.get("status_d") == "ok"
    wk_out, d_out = wk_ok and bool(r.get("outage_w")), d_ok and bool(r.get("outage_d"))
    if d_out and not r["censored_d"]:
        return r["outage_days"], False
    if d_out:
        return (max(DAILY_DAYS, r["recovery_weeks"] * 7), bool(r["censored_w"])) if wk_out else (r["outage_days"], True)
    if wk_out:
        return r["recovery_weeks"] * 7, bool(r["censored_w"])
    return (0.0, False) if (d_ok or wk_ok) else (np.nan, np.nan)

In [ ]:
# 4. Load Step 2b/3 outputs, typhoon tracks, municipal boundaries (with provinces) and RWI
import requests
import geopandas as gpd
import matplotlib.pyplot as plt

labels = pd.read_csv(f"{DRIVE_DIR}/step2b_labels.csv", encoding="utf-8-sig")
feat = pd.read_csv(f"{DRIVE_DIR}/step3_features.csv", encoding="utf-8-sig")
EMB = [c for c in feat.columns if re.fullmatch(r"A\d\d", c)]
print("step3_features:", feat.shape, "| embeddings:", len(EMB))

IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/"
               "v04r01/access/csv/ibtracs.WP.list.v04r01.csv")
ib = pd.read_csv(IBTRACS_URL, usecols=["SID", "SEASON", "NAME", "ISO_TIME", "LAT", "LON", "USA_WIND", "WMO_WIND", "USA_RMW"],
                 skiprows=[1], low_memory=False)
ib["SEASON"] = pd.to_numeric(ib["SEASON"], errors="coerce")
ib = ib[ib["SEASON"] >= 2017].copy()
for c in ["LAT", "LON", "USA_WIND", "WMO_WIND", "USA_RMW"]:
    ib[c] = pd.to_numeric(ib[c], errors="coerce")
ib["time_pht"] = pd.to_datetime(ib["ISO_TIME"]) + pd.Timedelta(hours=8)
tracks = {}
for ev in EVENTS.itertuples():
    t = ib[(ib.NAME == ev.ibtracs_name) & (ib.SEASON == ev.season)]
    if t.SID.nunique() > 1:
        t = t[t.SID == t.assign(g=(t.time_pht - ev.landfall).abs()).groupby("SID").g.min().idxmin()]
    tracks[ev.key] = t.sort_values("time_pht")

def load_gb(level):
    try:
        meta = requests.get(f"https://www.geoboundaries.org/api/current/gbOpen/PHL/{level}/", timeout=60).json()
        return gpd.read_file(meta["simplifiedGeometryGeoJSON"])
    except Exception as e:
        print(f"{level}: API failed, trying the GitHub mirror ({e})")
        return gpd.read_file("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/main/releaseData/"
                             f"gbOpen/PHL/{level}/geoBoundaries-PHL-{level}_simplified.geojson")

adm3 = load_gb("ADM3").to_crs(4326)[["shapeID", "shapeName", "geometry"]].rename(columns={"shapeName": "muni"})
adm2 = load_gb("ADM2").to_crs(4326)[["shapeName", "geometry"]].rename(columns={"shapeName": "province"})
pts = adm3.set_geometry(adm3.representative_point())
adm3 = adm3.merge(gpd.sjoin(pts, adm2, predicate="within", how="left")[["shapeID", "province"]]
                  .drop_duplicates("shapeID"), on="shapeID", how="left")
UTM = 32651
adm3_utm = adm3.to_crs(UTM)
anchor = adm3_utm.set_geometry(adm3_utm.representative_point())
anchor_ll = anchor.to_crs(4326)

rwi_files = glob.glob(f"{DRIVE_DIR}/*relative_wealth*.csv")
if rwi_files:
    rwi = pd.read_csv(rwi_files[0])
    rp = gpd.GeoDataFrame(rwi, geometry=gpd.points_from_xy(rwi.longitude, rwi.latitude), crs=4326)
    RWI = gpd.sjoin(rp, adm3[["shapeID", "geometry"]], predicate="within").groupby("shapeID").rwi.mean()
    print(f"RWI: {RWI.size} municipalities with values")
else:
    RWI = pd.Series(dtype=float, name="rwi")
    print("No RWI file; the wealth breakdown in Part D will be skipped")

INK, INK2, SURFACE, GRID = "#1f1f1e", "#5f5e5a", "#fcfcfb", "#e5e4e0"
C1, C2, C3, C4 = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
plt.rcParams.update({"font.size": 10, "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2,
                     "ytick.color": INK2, "axes.edgecolor": "#c9c8c4", "figure.facecolor": SURFACE,
                     "axes.facecolor": SURFACE})

## A. Placebo calibration

For each typhoon, compute the share with "outage > 7 days" at the real and the placebo landfall under five outage thresholds: 80%, 70%, 60%, 50% and 40%. A lower threshold is stricter: the lights must fall further to count as an outage.

Selection rule: **the most lenient threshold whose placebo "outage > 7 days" share is ≤ 5%**. If even 40% does not get there, take the threshold with the lowest placebo share and flag it.

Note: the placebo baseline falls in a different season (for Rai it is the August–October wet season), so its noise may be larger than in the real period. The calibration is therefore conservative: it accepts missing some real outages in exchange for fewer false positives.

In [ ]:
# 5. Recompute real and placebo labels at each threshold (about 5 minutes)
cal_rows = []
for key in MODEL_KEYS:
    ev = EV.loc[key].rename(key)
    d = load_event(key)
    meta = pd.read_csv(f"{DRIVE_DIR}/meta_{key}.csv")
    ids = set(feat.loc[feat.event == key, "shapeID"])
    d = d[d.shapeID.isin(ids)].merge(meta[["shapeID", "eligible_px"]], on="shapeID", how="left")
    bs, be = windows(ev)
    k = 1
    while ev.landfall - pd.Timedelta(days=k * SYNODIC) + pd.Timedelta(days=28) > be:
        k += 1
    t0p = ev.landfall - pd.Timedelta(days=k * SYNODIC)
    pb_end = t0p - pd.Timedelta(days=BASE_GAP_DAYS)
    for sid, g in d.groupby("shapeID"):
        for lvl in LEVELS:
            dr, cr = combine(label_all(g, ev.landfall, bs, be, pd.NaT, lvl))
            dp, _ = combine(label_all(g, t0p, bs, pb_end, be, lvl))
            cal_rows.append({"event": key, "shapeID": sid, "level": lvl, "real_days": dr, "real_cens": cr,
                             "placebo_days": dp})
    print(key, "done")

cal = pd.DataFrame(cal_rows).merge(feat[["event", "shapeID", "dist_km"]], on=["event", "shapeID"], how="left")
def long_share(days, cens=None):
    ok = days.notna()
    if cens is not None:
        ok &= ~((cens == True) & (days <= LONG_OUTAGE_DAYS))
    return (days[ok] > LONG_OUTAGE_DAYS).mean()

CAL = cal.groupby(["event", "level"]).apply(lambda x: pd.Series({
    "placebo > 7 d": long_share(x.placebo_days),
    "real > 7 d (all)": long_share(x.real_days, x.real_cens),
    "real > 7 d (≤30 km)": long_share(x.real_days[x.dist_km <= 30], x.real_cens[x.dist_km <= 30]),
    "real > 7 d (>100 km)": long_share(x.real_days[x.dist_km > 100], x.real_cens[x.dist_km > 100]),
}), include_groups=False)

chosen = {}
for key in MODEL_KEYS:
    s = CAL.loc[key, "placebo > 7 d"]
    okl = s[s <= TARGET_FP]
    chosen[key] = (int(okl.index.max()), "") if len(okl) else (int(s.idxmin()), "WARNING: target not reached")
print({k: v[0] for k, v in chosen.items()})
(CAL * 100).round(1)

In [ ]:
# 6. Figure: calibration curves, and build the calibrated labels
fig, axes = plt.subplots(1, len(MODEL_KEYS), figsize=(3.8 * len(MODEL_KEYS), 3.4), sharey=True, constrained_layout=True)
for ax, key in zip(axes, MODEL_KEYS):
    c = CAL.loc[key] * 100
    ax.plot(c.index, c["real > 7 d (≤30 km)"], color=C1, lw=2, marker="o", ms=5, label="Real, ≤ 30 km")
    ax.plot(c.index, c["real > 7 d (>100 km)"], color=C1, lw=1.5, ls="--", marker="o", ms=4, label="Real, > 100 km")
    ax.plot(c.index, c["placebo > 7 d"], color="#a3a29d", lw=2, marker="o", ms=5, label="Placebo")
    ax.axhline(TARGET_FP * 100, color=INK2, lw=0.8, ls=":")
    lvl, flag = chosen[key]
    ax.axvline(lvl, color=C2, lw=1.2)
    ax.annotate(f"chosen {lvl}%{' ' + flag if flag else ''}", (lvl, 40), xytext=(4, 0), textcoords="offset points",
                fontsize=8, color=INK2, va="center", bbox=dict(boxstyle="round,pad=0.2", fc=SURFACE, ec="none"))
    ax.set_xlim(85, 35)
    ax.set_ylim(0, 100)
    ax.set_title(key, loc="left", fontsize=10)
    ax.set_xlabel("Outage threshold (% of baseline)")
    ax.grid(axis="y", color=GRID, lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[0].set_ylabel(f"Share with outage > {LONG_OUTAGE_DAYS} days (%)")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=3, frameon=False, fontsize=9, bbox_to_anchor=(0.5, -0.1))
fig.suptitle(f"Placebo calibration: the most lenient threshold with a placebo rate ≤ {TARGET_FP:.0%} (dotted line)",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step4_calibration.png", dpi=200, bbox_inches="tight")
plt.show()

pick = cal[cal.apply(lambda r: r.level == chosen[r.event][0], axis=1)]
pick = pick[~((pick.real_cens == True) & (pick.real_days <= LONG_OUTAGE_DAYS))].dropna(subset=["real_days"])
ycal = pick.assign(y_cal=(pick.real_days > LONG_OUTAGE_DAYS).astype(int))[["event", "shapeID", "y_cal", "level"]]
feat = feat.drop(columns=[c for c in ["y_cal", "level"] if c in feat.columns]).merge(ycal, on=["event", "shapeID"], how="left")
print(pd.DataFrame({"step 3 label (> 7 d)": feat.groupby("event").y.mean(),
                    "calibrated label": feat.groupby("event").y_cal.mean(),
                    "agreement": feat.assign(a=feat.y == feat.y_cal).groupby("event").a.mean()}).round(2))

## B. Spatially explicit features

| Feature | Meaning | Why it might matter |
| --- | --- | --- |
| `wind_lag6` | Mean maximum wind of the 6 nearest municipalities | Distribution lines cross municipal borders, so damage next door slows local restoration |
| `log_island_km2` | Area of the island the municipality is on (log) | Small islands usually have separate or fragile grids, and crews and materials must cross the sea |
| `coop_mean_wind` | Mean wind over the utility's whole service territory | A cooperative shares its restoration crews; the harder the whole territory is hit, the thinner they are spread |
| `coop_share_typhoon` | Share of the territory's municipalities with wind ≥ 33 m/s (typhoon force) | As above, but counting only the worst-hit part |
| `log_coop_customers` | Number of customers of the utility (log) | Size roughly reflects technical and financial capacity |
| `coop_private` | Whether the utility is private (PIOU) | Private utilities and cooperatives differ in resources and incentives |

**Matching service areas**: Wikipedia lists municipality names without provinces, and many names repeat (San Isidro, for example). The matching rules:
1. Unique name: match directly
2. Repeated name: prefer the one in the same province as the utility's other municipalities; if several remain, take the one closest to them

**Manual fallback**: if parsing fails, put a `utility_areas.csv` in Drive with columns `utility, type, customers, municipality, province`; this cell reads it first.

In [ ]:
# 7. Read utility service areas and match them to municipalities
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_electric_distribution_utilities_in_the_Philippines"

def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"\[.*?\]|\(.*?\)", " ", s)
    for a, b in [(r"\bsto\.?\s", "santo "), (r"\bsta\.?\s", "santa "), (r"\bgen\.?\s", "general "),
                 (r"\bpres\.?\s", "president "), (r"\bcity of\b", " "), (r"\bcity\b", " "),
                 (r"\bmunicipality of\b", " ")]:
        s = re.sub(a, b, s + " ")
    return re.sub(r"[^a-z0-9]+", " ", s).strip()

manual = f"{DRIVE_DIR}/utility_areas.csv"
if os.path.exists(manual):
    ua = pd.read_csv(manual)
    ua.columns = [c.strip().lower() for c in ua.columns]
    print("Using the manual file utility_areas.csv:", ua.shape)
else:
    html = requests.get(WIKI_URL, headers={"User-Agent": "Mozilla/5.0 (research notebook; typhoon outage study)"},
                        timeout=60).text
    # Key step: read_html merges line breaks into spaces, so turn list items and line-break tags into semicolons first
    html = re.sub(r"(?i)</li>|</p>|<br\s*/?>", ";", html)
    tables = pd.read_html(io.StringIO(html), match="Areas served")
    t = max(tables, key=len)
    t.columns = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in t.columns]
    col = lambda key: next(c for c in t.columns if key in c.lower())
    rows = []
    n_ok = 0
    for r in t.itertuples(index=False):
        rec = dict(zip(t.columns, r))
        text = str(rec[col("areas")])
        m = re.search(r"List\s*\((\d+)\)", text)
        items = [x.strip() for x in re.split(r"[;\n•·]", re.sub(r"List\s*\(\d+\)", "", text))]
        items = [re.sub(r"\[.*?\]", "", x).strip() for x in items if x.strip() and x.strip().lower() != "nan"]
        n_ok += bool(m and int(m.group(1)) == len(items))
        cust = pd.to_numeric(re.sub(r"[^\d]", "", str(rec[col("customers")])) or np.nan, errors="coerce")
        for it in items:
            rows.append({"utility": rec[col("acronym")], "utility_name": rec[col("utility")],
                         "type": rec[col("type")], "grid": rec[col("grid")], "customers": cust,
                         "municipality": it, "province": np.nan})
    ua = pd.DataFrame(rows)
    print(f"Parsed from Wikipedia: {ua.utility.nunique()} utilities, {len(ua)} utility–municipality pairs; "
          f"share of utilities whose item count matches 'List (N)': {n_ok / len(t):.0%}")
    ua["utility"] = ua.utility.where(ua.utility.notna() & (ua.utility.astype(str) != "nan"), ua.utility_name)
    ua.to_csv(f"{DRIVE_DIR}/utility_areas_parsed.csv", index=False, encoding="utf-8-sig")

adm3["key"] = adm3.muni.map(norm)
cand = adm3.groupby("key").apply(lambda x: x[["shapeID", "province"]].to_dict("records"), include_groups=False).to_dict()

# If a whole item has no match and contains commas (e.g. "Baybay, Albuera"), split on commas and try again
def expand(r):
    parts = [x.strip() for x in str(r.municipality).split(",") if x.strip()]
    if norm(r.municipality) in cand or len(parts) < 2:
        return [r.municipality]
    return parts
ua = ua.assign(municipality=ua.apply(expand, axis=1)).explode("municipality").reset_index(drop=True)
xy = anchor.set_index("shapeID").geometry

assign_rows, unmatched = [], []
for u, g in ua.groupby("utility"):
    first = [cand.get(norm(m), []) for m in g.municipality]
    # The utility's home province(s): those containing the most of its listed names (several if it spans provinces)
    votes = pd.Series([p for cs in first for p in {c["province"] for c in cs if isinstance(c["province"], str)}])
    provs = votes.value_counts()
    provs = provs[provs >= 0.5 * provs.max()].index.tolist() if len(provs) else []
    in_home = [[c for c in cs if c["province"] in provs] or cs for cs in first]
    uniq = [cs[0] for cs in in_home if len(cs) == 1]
    ref = uniq or [c for cs in in_home for c in cs]      # without unique matches, use the centre of all candidates
    centre = (np.mean([xy[c["shapeID"]].x for c in ref]), np.mean([xy[c["shapeID"]].y for c in ref])) if ref else None
    for (_, r), cs in zip(g.iterrows(), in_home):
        if isinstance(r.province, str):
            cs = [c for c in cand.get(norm(r.municipality), []) if norm(c["province"]) == norm(r.province)] or cs
        if len(cs) > 1 and centre is not None:
            cs = [min(cs, key=lambda c: (xy[c["shapeID"]].x - centre[0]) ** 2 + (xy[c["shapeID"]].y - centre[1]) ** 2)]
        if len(cs) == 1:
            assign_rows.append({"shapeID": cs[0]["shapeID"], "utility": u, "type": r.type, "customers": r.customers})
        else:
            unmatched.append((u, r.municipality, len(cs)))

UA = pd.DataFrame(assign_rows)
n_util = UA.groupby("shapeID").utility.nunique()
UA1 = UA.drop_duplicates("shapeID")   # when a municipality has several utilities, keep the first
cov = feat.shapeID.isin(UA1.shapeID).mean()
print(f"Share of modelling municipalities matched to a utility: {cov:.0%}; "
      f"municipalities with several utilities: {(n_util > 1).sum()}; "
      f"unmatched names: {len(unmatched)}")
size = UA1.groupby("utility").size()
print(f"Municipalities matched per utility: median {size.median():.0f}, "
      f"utilities with only 1 matched municipality: {(size == 1).sum()} / {len(size)}")
print((feat.assign(m=feat.shapeID.isin(UA1.shapeID)).groupby("event").m.mean() * 100).round(0).rename("matched (%)").to_string())
if cov < 0.8 or size.median() < 3:
    print("WARNING: "
          "Low match rate, or most utilities matched only 1 municipality. Inspect utility_areas_parsed.csv in Drive, tidy it into utility_areas.csv and rerun this cell")
print("Unmatched examples:", unmatched[:15])

In [ ]:
# 8. Compute the spatially explicit features
from sklearn.neighbors import NearestNeighbors
KT2MS, NMI2KM, R_EARTH = 0.514444, 1.852, 6371.0

def haversine_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1)[:, None], np.radians(lat2)[None, :]
    dphi, dl = p2 - p1, np.radians(lon2)[None, :] - np.radians(lon1)[:, None]
    a = np.sin(dphi / 2) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2) ** 2
    return 2 * R_EARTH * np.arcsin(np.sqrt(a))

def wind_max(track, landfall, lat, lon):
    t = track[(track.time_pht - landfall).abs() <= pd.Timedelta(days=3)].copy()
    t["vmax"] = (t.USA_WIND * KT2MS).fillna(t.WMO_WIND / 0.88 * KT2MS)
    t["rmw"] = t.USA_RMW * NMI2KM
    t = (t.drop_duplicates("time_pht").set_index("time_pht")[["LAT", "LON", "vmax", "rmw"]]
           .resample("30min").mean().interpolate(limit_direction="both"))
    t["rmw"] = t["rmw"].fillna(40).clip(10, 100)
    r = haversine_km(lat, lon, t.LAT.values, t.LON.values)
    R, V = t.rmw.values[None, :], t.vmax.values[None, :]
    return np.where(r < R, V * r / R, V * (R / np.maximum(r, 1e-6)) ** 0.5).max(axis=1)

# Islands: buffer municipal boundaries by 200 m, merge, split into separate landmasses
buf = adm3_utm.buffer(200)
land = gpd.GeoSeries(buf.union_all() if hasattr(buf, "union_all") else buf.unary_union, crs=UTM).explode(index_parts=False).reset_index(drop=True)
isl = gpd.GeoDataFrame({"island_km2": land.area / 1e6}, geometry=land, crs=UTM)
island = gpd.sjoin(anchor[["shapeID", "geometry"]], isl, predicate="within", how="left").drop_duplicates("shapeID")
island = island.set_index("shapeID").island_km2

nbr = NearestNeighbors(n_neighbors=7).fit(np.c_[anchor.geometry.x, anchor.geometry.y])
nbr_idx = nbr.kneighbors(np.c_[anchor.geometry.x, anchor.geometry.y], return_distance=False)[:, 1:]
util_of = UA1.set_index("shapeID")

sp_rows = []
for key in MODEL_KEYS:
    w = wind_max(tracks[key], EV.loc[key, "landfall"], anchor_ll.geometry.y.values, anchor_ll.geometry.x.values)
    W = pd.Series(w, index=anchor.shapeID.values)
    lag = pd.Series(w[nbr_idx].mean(axis=1), index=anchor.shapeID.values)
    cw = UA1.assign(w=UA1.shapeID.map(W)).groupby("utility").w.agg(
        coop_mean_wind="mean", coop_share_typhoon=lambda s: (s >= 33).mean())
    ids = feat.loc[feat.event == key, "shapeID"]
    u = util_of.reindex(ids)
    sp_rows.append(pd.DataFrame({
        "event": key, "shapeID": ids.values,
        "wind_lag6": lag.reindex(ids).values,
        "log_island_km2": np.log(island.reindex(ids).values),
        "coop_mean_wind": cw.coop_mean_wind.reindex(u.utility).values,
        "coop_share_typhoon": cw.coop_share_typhoon.reindex(u.utility).values,
        "log_coop_customers": np.log(u.customers.astype(float)).values,
        "coop_private": u.type.astype(str).str.upper().str.contains("PIOU").astype(float).where(u.type.notna()).values,
        "utility": u.utility.values}))
SP = pd.concat(sp_rows, ignore_index=True)
feat = feat.drop(columns=[c for c in SP.columns if c in feat.columns and c not in ("event", "shapeID")]).merge(
    SP, on=["event", "shapeID"], how="left")
SPATIAL_GEO = ["wind_lag6", "log_island_km2"]
SPATIAL_COOP = ["coop_mean_wind", "coop_share_typhoon", "log_coop_customers", "coop_private"]
SPATIAL = SPATIAL_GEO + SPATIAL_COOP
print(feat[SPATIAL].describe().T[["count", "mean", "min", "max"]].round(2))
if feat.coop_mean_wind.notna().mean() < 0.5:
    print("WARNING: Fewer than half of municipalities have cooperative features, so H+S mostly reflects H+Sgeo")

## C. Model comparison with confidence intervals

The main analysis uses the **calibrated labels**. Seven feature sets:

| Name | Contents |
| --- | --- |
| H | Hazard (wind, gale hours, distance to track, rainfall) |
| H+Sgeo | Hazard + neighbouring wind + island size |
| H+S | Hazard + all spatially explicit features (adds the 4 cooperative features) |
| H+E | Hazard + exposure (population, built-up, pre-event light, elevation, distance to coast, area) |
| H+E+S | Hazard + exposure + spatial |
| H+E+Emb3 | Hazard + exposure + embeddings (3 principal components) |
| H+E+Emb10 | Hazard + exposure + embeddings (10 principal components, as in Step 3) |

**Confidence intervals**: bootstrap the leave-one-typhoon-out predictions. Each replicate resamples municipalities with replacement within each typhoon and recomputes the mean AUC over the 4 typhoons; 1000 replicates. **Differences** from H use the same resamples (paired bootstrap). Only when the 95% interval of a difference excludes 0 can we say two feature sets really differ.

For comparison, leave-one-typhoon-out is also run on the original Step 3 labels.

In [ ]:
# 9. Train and evaluate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GroupKFold, LeaveOneGroupOut
from sklearn.metrics import roc_auc_score

HAZARD = ["wind_ms", "gale_hours", "dist_km", "rain_mm"]
EXPOSURE = ["log_pop_density", "built_frac", "log_pre_ntl", "elev_m", "dist_coast_km", "log_area_km2"]
SETS = {"H": (HAZARD, 0), "H+Sgeo": (HAZARD + SPATIAL_GEO, 0), "H+S": (HAZARD + SPATIAL, 0), "H+E": (HAZARD + EXPOSURE, 0),
        "H+E+S": (HAZARD + EXPOSURE + SPATIAL, 0),
        "H+E+Emb3": (HAZARD + EXPOSURE + EMB, 3), "H+E+Emb10": (HAZARD + EXPOSURE + EMB, 10)}
MODELS = ["Logistic regression", "Gradient boosting"]
SCHEMES = ["Random split", "Spatial blocks (province)", "Leave one typhoon out"]

def make_model(kind, cols, n_pca):
    emb = [c for c in cols if c in EMB]
    other = [c for c in cols if c not in EMB]
    parts = [("other", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), other)]
    if emb and n_pca:
        parts.append(("emb", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                       ("pca", PCA(n_components=n_pca, random_state=0))]), emb))
    clf = (LogisticRegression(C=1.0, max_iter=5000) if kind == MODELS[0] else
           HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200,
                                          l2_regularization=1.0, random_state=0))
    return Pipeline([("pre", ColumnTransformer(parts)), ("clf", clf)])

def split_iter(scheme, data, seed):
    y = data.target.values
    if scheme == SCHEMES[0]:
        return StratifiedKFold(5, shuffle=True, random_state=seed).split(data, y)
    if scheme == SCHEMES[1]:
        return GroupKFold(5).split(data, y, groups=data.province.fillna("unknown"))
    return LeaveOneGroupOut().split(data, y, groups=data.event)

fold_rows, oof = [], {}
for label_name, ycol, schemes in [("calibrated", "y_cal", SCHEMES), ("step 3", "y", SCHEMES[2:])]:
    data = feat.dropna(subset=[ycol]).assign(target=lambda x: x[ycol].astype(int)).reset_index(drop=True)
    for scheme in schemes:
        for kind in MODELS:
            for set_name, (cols, n_pca) in SETS.items():
                for seed in ([0, 1, 2] if scheme == SCHEMES[0] else [0]):
                    pred = np.full(len(data), np.nan)
                    for tr, te in split_iter(scheme, data, seed):
                        m = make_model(kind, cols, n_pca).fit(data.loc[tr, cols], data.target[tr])
                        pred[te] = m.predict_proba(data.loc[te, cols])[:, 1]
                        yt = data.target[te]
                        fold_rows.append({"label": label_name, "scheme": scheme, "model": kind, "set": set_name,
                                          "seed": seed, "held_out": data.event[te].iloc[0] if scheme == SCHEMES[2] else "",
                                          "auc": roc_auc_score(yt, pred[te]) if yt.nunique() == 2 else np.nan})
                    if seed == 0:
                        oof[(label_name, scheme, kind, set_name)] = (data, pred)
        print(label_name, scheme, "done")

folds = pd.DataFrame(fold_rows)
summary = folds.groupby(["label", "scheme", "model", "set"], sort=False).auc.agg(["mean", "std"]).round(3)
summary

In [ ]:
# 10. Bootstrap confidence intervals: mean leave-one-typhoon-out AUC, and difference from H
rng = np.random.default_rng(0)

def event_mean_auc(y, p, ev, idx):
    aucs = []
    for e in np.unique(ev):
        i = idx[ev[idx] == e]
        if len(np.unique(y[i])) == 2:
            aucs.append(roc_auc_score(y[i], p[i]))
    return np.mean(aucs)

boot_rows = []
for label_name in ["calibrated", "step 3"]:
    for kind in MODELS:
        data, _ = oof[(label_name, SCHEMES[2], kind, "H")]
        y, ev = data.target.values, data.event.values
        groups = [np.where(ev == e)[0] for e in np.unique(ev)]
        draws = [np.concatenate([rng.choice(g, len(g), replace=True) for g in groups]) for _ in range(N_BOOT)]
        base_p = oof[(label_name, SCHEMES[2], kind, "H")][1]
        base_b = np.array([event_mean_auc(y, base_p, ev, d) for d in draws])
        for set_name in SETS:
            p = oof[(label_name, SCHEMES[2], kind, set_name)][1]
            b = np.array([event_mean_auc(y, p, ev, d) for d in draws]) if set_name != "H" else base_b
            diff = b - base_b
            rnd = folds[(folds.label == "calibrated") & (folds.scheme == SCHEMES[0]) & (folds.model == kind) &
                        (folds.set == set_name)].auc.mean() if label_name == "calibrated" else np.nan
            boot_rows.append({"label": label_name, "model": kind, "set": set_name,
                              "LOTO AUC": event_mean_auc(y, p, ev, np.arange(len(y))),
                              "CI low": np.percentile(b, 2.5), "CI high": np.percentile(b, 97.5),
                              "diff vs H": np.mean(diff), "diff CI low": np.percentile(diff, 2.5),
                              "diff CI high": np.percentile(diff, 97.5), "random-split AUC": rnd})
BOOT = pd.DataFrame(boot_rows)
BOOT["transfer loss"] = BOOT["random-split AUC"] - BOOT["LOTO AUC"]
BOOT["differs from H?"] = np.where(BOOT["diff CI low"] > 0, "better", np.where(BOOT["diff CI high"] < 0, "worse", "no clear difference"))
BOOT.loc[BOOT.set == "H", "differs from H?"] = "—"
BOOT.round(3)

In [ ]:
# 11. Figure: leave-one-typhoon-out AUC per feature set (95% CI); open circle = random split
set_order = list(SETS)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True, constrained_layout=True)
for ax, kind in zip(axes, MODELS):
    b = BOOT[(BOOT.label == "calibrated") & (BOOT.model == kind)].set_index("set").reindex(set_order)
    yy = np.arange(len(set_order))
    ax.hlines(yy, b["CI low"], b["CI high"], color=C1, lw=2)
    ax.plot(b["LOTO AUC"], yy, "o", color=C1, ms=8, label="Leave one typhoon out (95% CI)")
    ax.plot(b["random-split AUC"], yy, "o", mfc="none", mec=INK2, ms=8, label="Random split")
    ax.axvline(0.5, color="#a3a29d", lw=1, ls="--")
    ax.axvline(b.loc["H", "LOTO AUC"], color=C1, lw=0.8, ls=":")
    ax.set_yticks(yy, set_order)
    ax.invert_yaxis()
    lo = min(0.45, BOOT["CI low"].min() - 0.03)
    hi = min(1.0, max(BOOT["CI high"].max(), BOOT["random-split AUC"].max()) + 0.03)
    ax.set_xlim(lo, hi)
    ax.set_xlabel("AUC (mean over held-out typhoons)")
    ax.set_title(kind, loc="left", fontsize=11)
    ax.grid(axis="x", color=GRID, lw=0.8)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)
axes[1].legend(frameon=False, fontsize=9, loc="lower right")
fig.suptitle("Transfer to an unseen typhoon, calibrated labels. Dotted line = hazard-only model",
             x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step4_auc_ci.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 12. Residual diagnostics: Moran's I, and whether residuals cluster by utility or province
def morans_i(z, xy, k=6, n_perm=499, seed=0):
    idx = NearestNeighbors(n_neighbors=k + 1).fit(xy).kneighbors(xy, return_distance=False)[:, 1:]
    z = z - z.mean()
    stat = lambda v: (v * v[idx].mean(axis=1)).sum() / (v ** 2).sum()
    obs = stat(z)
    r = np.random.default_rng(seed)
    perm = np.array([stat(r.permutation(z)) for _ in range(n_perm)])
    return obs, (1 + (perm >= obs).sum()) / (n_perm + 1)

def eta2(res, groups, n_perm=499, seed=0):
    # share of variance between groups; permutation gives the chance level
    df_ = pd.DataFrame({"r": res, "g": groups}).dropna()
    df_ = df_[df_.groupby("g").g.transform("size") >= 2]   # single-municipality groups inflate η²; drop them
    if df_.g.nunique() < 2:
        return np.nan, np.nan, df_.g.nunique()
    f = lambda g: ((df_.groupby(g).r.transform("mean") - df_.r.mean()) ** 2).sum() / ((df_.r - df_.r.mean()) ** 2).sum()
    obs = f(df_.g)
    r = np.random.default_rng(seed)
    perm = np.array([f(pd.Series(r.permutation(df_.g.values), index=df_.index)) for _ in range(n_perm)])
    return obs, (1 + (perm >= obs).sum()) / (n_perm + 1), df_.g.nunique()

diag = []
kind = MODELS[0]
for set_name in ["H", "H+Sgeo", "H+S", "H+E", "H+E+S", "H+E+Emb10"]:
    data, p = oof[("calibrated", SCHEMES[2], kind, set_name)]
    d = data.assign(res=data.target - p)
    for e, g in d.groupby("event"):
        I, pI = morans_i(g.res.values, g[["x_utm", "y_utm"]].values)
        eu, pu, nu = eta2(g.res.values, g.utility.values)
        epv, pp, npv = eta2(g.res.values, g.province.values)
        diag.append({"set": set_name, "event": e, "Moran's I": I, "p (I)": pI,
                     "η² utility": eu, "p (utility)": pu, "utilities (n)": nu,
                     "η² province": epv, "p (province)": pp, "provinces (n)": npv})
DIAG = pd.DataFrame(diag)
print(f"{kind}, calibrated labels, leave-one-typhoon-out residuals")
DIAG.pivot_table(index="set", columns="event", values="Moran's I", sort=False).round(3)

In [ ]:
DIAG[DIAG.set == "H"].round(3)

## D. Equity: who can be measured, and who gets prioritised

1. **Measurement coverage**: among all affected municipalities in the 4 typhoons, compare by RWI tercile the share that satellites can measure at all. Municipalities with too little light have no label
2. **Priority recall**: within each typhoon, flag the 30% of municipalities with the highest predicted risk as priorities, and compute what share of municipalities with a true outage > 7 days are covered, by RWI tercile (for H and for the best-performing feature set)

Without an RWI file, pre-event light terciles are used instead.

In [ ]:
# 13. Measurement coverage and priority recall
swath = labels[labels.event.isin(MODEL_KEYS)][["event", "shapeID", "duration_source"]].copy()
swath["measured"] = swath.duration_source != "none"
if RWI.size:
    swath["wealth"] = swath.shapeID.map(RWI)
    GROUP_LABELS, group_name = ["Poorest third", "Middle", "Richest third"], "RWI tercile"
else:
    pre = feat.groupby("shapeID").log_pre_ntl.mean()
    swath["wealth"] = swath.shapeID.map(pre)
    GROUP_LABELS, group_name = ["Dimmest third", "Middle", "Brightest third"], "pre-event light tercile"
swath = swath.dropna(subset=["wealth"])
cut = np.quantile(swath.wealth, [1 / 3, 2 / 3])
tercile = lambda v: pd.cut(v, [-np.inf, cut[0], cut[1], np.inf], labels=GROUP_LABELS)
swath["group"] = tercile(swath.wealth)
COVER = swath.groupby("group", observed=True).measured.agg(["size", "mean"]).rename(
    columns={"size": "municipalities in swath", "mean": "share measurable"})
print(f"By {group_name}:")
display(COVER.round(3))

best = (BOOT[(BOOT.label == "calibrated") & (BOOT.model == MODELS[0])].sort_values("LOTO AUC", ascending=False).set.iloc[0])
rec_rows = []
for set_name in dict.fromkeys(["H", best]):
    data, p = oof[("calibrated", SCHEMES[2], MODELS[0], set_name)]
    d = data.assign(p=p)
    d["flag"] = d.groupby("event").p.transform(lambda s: s.rank(ascending=False, method="first") <= np.ceil(PRIORITY_SHARE * len(s)))
    d["group"] = tercile(d.shapeID.map(RWI) if RWI.size else d.log_pre_ntl)
    pos = d[d.target == 1].dropna(subset=["group"])
    for grp, g in list(pos.groupby("group", observed=True)) + [("All", pos)]:
        f = g.flag.values.astype(float)
        bs = [rng.choice(f, len(f), replace=True).mean() for _ in range(N_BOOT)]
        rec_rows.append({"set": set_name, "group": grp, "long outages": len(g), "recall@30%": f.mean(),
                         "CI low": np.percentile(bs, 2.5), "CI high": np.percentile(bs, 97.5)})
REC = pd.DataFrame(rec_rows)
print(f"\nRecall when prioritising the top {PRIORITY_SHARE:.0%} (logistic regression, calibrated labels):")
display(REC.round(3))

In [ ]:
# 14. Figure: equity, two panels
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8), constrained_layout=True)
ax = axes[0]
ax.barh(range(len(COVER)), COVER["share measurable"] * 100, color=C1, height=0.6)
for i, v in enumerate(COVER["share measurable"] * 100):
    ax.text(v + 1, i, f"{v:.0f}%", va="center", fontsize=9, color=INK)
ax.set_yticks(range(len(COVER)), COVER.index)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Share of affected municipalities with a satellite label (%)")
ax.set_title(f"Who can be measured ({group_name})", loc="left", fontsize=11)
ax = axes[1]
sets_shown = REC.set.unique().tolist()
colors = dict(zip(sets_shown, [C1, C2]))
groups_ = GROUP_LABELS + ["All"]
for j, s in enumerate(sets_shown):
    r = REC[REC.set == s].set_index("group").reindex(groups_)
    yy = np.arange(len(groups_)) + (j - (len(sets_shown) - 1) / 2) * 0.22
    ax.hlines(yy, r["CI low"] * 100, r["CI high"] * 100, color=colors[s], lw=2)
    ax.plot(r["recall@30%"] * 100, yy, "o", color=colors[s], ms=7, label=s)
ax.axvline(PRIORITY_SHARE * 100, color="#a3a29d", lw=1, ls="--")
ax.text(PRIORITY_SHARE * 100 + 1, -0.45, "random ranking", fontsize=8, color=INK2, va="center")
ax.set_yticks(range(len(groups_)), groups_)
ax.set_ylim(len(groups_) - 0.5, -0.7)
ax.set_xlim(0, 100)
ax.set_xlabel(f"Long outages covered when prioritising the top {PRIORITY_SHARE:.0%} (%, 95% CI)")
ax.set_title("Who gets prioritised (leave one typhoon out)", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=9, loc="lower right")
for a in axes:
    a.grid(axis="x", color=GRID, lw=0.8)
    for s in ("top", "right"):
        a.spines[s].set_visible(False)
fig.suptitle("Equity: measurement coverage and priority recall", x=0.01, ha="left", fontsize=12)
fig.savefig("figures/step4_equity.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# 15. Save and download
feat.to_csv(f"{DRIVE_DIR}/step4_features.csv", index=False, encoding="utf-8-sig")
out = {"step4_calibration.csv": (CAL * 100).round(2).reset_index(), "step4_bootstrap.csv": BOOT.round(4),
       "step4_folds.csv": folds, "step4_residual_diagnostics.csv": DIAG.round(4), "step4_recall.csv": REC.round(4),
       "step4_coverage.csv": COVER.round(4).reset_index()}
for name, t in out.items():
    t.to_csv(f"figures/{name}", index=False, encoding="utf-8-sig")
    t.to_csv(f"{DRIVE_DIR}/{name}", index=False, encoding="utf-8-sig")
from google.colab import files
for f in list(out) + ["step4_calibration.png", "step4_auc_ci.png", "step4_equity.png"]:
    files.download(f"figures/{f}")